# Chatbot RAG — Text Mining (versione semplificata)

Domande in italiano su uno o più PDF, con riferimenti a file e pagina.

Questo notebook usa **la stessa logica dell'app Streamlit** (pacchetto `rag/`), così i due
restano allineati: qui ogni passaggio della pipeline è eseguito e mostrato separatamente.

- **In locale** (Jupyter / VS Code): apri il notebook dalla cartella `notebook/` del progetto.
- **Su Google Colab**: la prima cella chiede di caricare lo zip del progetto (che contiene
  `rag/`). La GPU non serve: la generazione avviene via API (Groq).

Versione semplificata: un solo servizio (Groq) e domande indipendenti, senza memoria della conversazione.

Pipeline: PDF → pagine → chunk → indice ibrido (BGE-M3 + TF-IDF, fusi con RRF) →
passaggi → LLM via API (Groq) → risposta con fonti.

## 1. Ambiente

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip -q install sentence-transformers==5.1.2 transformers==4.57.1 pymupdf==1.26.6 scikit-learn==1.7.2 groq python-dotenv
    if not Path("rag").exists():
        from google.colab import files
        print("Carica lo zip del progetto (deve contenere la cartella rag/).")
        for nome in files.upload():
            zipfile.ZipFile(nome).extractall(".")
    # Lo zip può contenere la cartella del progetto: cerco dove sta rag/
    radice = next((p.parent for p in Path(".").rglob("rag/__init__.py")), None)
    radice = radice.parent if radice else Path(".")
else:
    radice = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

sys.path.insert(0, str(radice.resolve()))
import rag  # noqa: F401  (sistema anche i certificati SSL su Windows)
print("Progetto:", radice.resolve())

## 2. Chiavi API
La chiave gratuita si ottiene su [Groq](https://console.groq.com/keys). Non va mai scritta nel notebook:
- in locale sta nel file `.env` del progetto (vedi `.env.example`);
- su Colab nei **Secrets** (icona a forma di chiave), con il nome `GROQ_API_KEY`.

In [ ]:
from dotenv import load_dotenv

if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        pass
else:
    load_dotenv(radice / ".env")

print("GROQ_API_KEY", "presente" if os.environ.get("GROQ_API_KEY") else "assente")

## 3. Lettura dei PDF
Il testo viene estratto per blocchi, conservando le righe e i numeri di pagina.
Le pagine senza testo (scansioni) vengono segnalate: servirebbe l'OCR.

In [ ]:
from rag.pdf import estrai_pagine

if IN_COLAB:
    from google.colab import files
    caricati = files.upload()
    pdf = {nome: dati for nome, dati in caricati.items() if nome.lower().endswith(".pdf")}
else:
    # Indica qui i PDF da usare (percorsi assoluti o relativi alla cartella del notebook).
    PERCORSI = sorted((radice / "pdf").glob("*.pdf"))
    pdf = {Path(p).name: Path(p).read_bytes() for p in PERCORSI}

if not pdf:
    raise ValueError("Nessun PDF: caricane almeno uno (in locale, mettili nella cartella pdf/).")

for nome, dati in pdf.items():
    pagine, vuote = estrai_pagine(nome, dati)
    print(f"{nome}: {len(pagine)} pagine con testo" + (f", senza testo: {vuote}" if vuote else ""))

## 4. Chunk, embedding e indice ibrido
`BAAI/bge-m3` per la ricerca semantica multilingue (sulla GPU se c'è, altrimenti CPU).
Le pagine brevi restano intere; quelle lunghe vengono divise in finestre di 450 token con
75 token di sovrapposizione, senza unire pagine diverse. Gli embedding vengono salvati in
`.cache/` e non vengono ricalcolati se si ricarica lo stesso file.

In [ ]:
from rag.index import Indice, carica_embedder, indicizza_documento

embedder = carica_embedder()
documenti = [indicizza_documento(nome, dati, embedder, radice / ".cache" / "embeddings")
             for nome, dati in pdf.items()]
indice = Indice(documenti, embedder)
for d in documenti:
    print(f"{d.nome}: {d.n_pagine} pagine, {len(d.chunks)} chunk")
print("Totale chunk:", len(indice.chunks), "| dispositivo:", embedder.device)

## 5. Recupero dei passaggi
Ranking semantico (coseno tra embedding) e lessicale (TF-IDF su unigrammi e bigrammi,
con i mesi italiani e inglesi normalizzati) fusi con **Reciprocal Rank Fusion** pesata
(0,8 semantico, 0,2 lessicale, k = 60).

In [ ]:
DOMANDA_PROVA = "Quando si presentano i progetti?"

for i, r in enumerate(indice.cerca(DOMANDA_PROVA), start=1):
    anteprima = r["testo"][:110].replace("\n", " ")
    print(f"[{i}] {r['file']} p.{r['pagina']}  RRF {r['punteggio']:.4f}  "
          f"sem {r['semantico']:.3f}  lex {r['lessicale']:.3f}\n    {anteprima}…")

### Valutazione del retrieval
Le 12 prove sul PDF `TextMining_2026-27_DayOne.pdf` (in `eval.py`): una prova passa se
almeno una delle pagine attese è tra i primi quattro risultati.

In [ ]:
from eval import valuta_retrieval

if len(pdf) == 1 and "dayone" in next(iter(pdf)).lower().replace("_", ""):
    valuta_retrieval(indice, top_k=4)
else:
    print("Le prove predefinite richiedono il solo PDF DayOne.")

## 6. Modello generativo via API (Groq)
Il menu dei modelli disponibili viene letto dall'API. Per i modelli `gpt-oss`, che ragionano
prima di rispondere, il ragionamento è impostato al minimo: per estrarre una risposta da
pochi passaggi basta, ed è molto più veloce.

In [ ]:
from rag import config
from rag.llm import GroqLLM, chiave_da_ambiente, scegli_modello

MODELLO = None  # None = il modello predefinito tra quelli disponibili

chiave = chiave_da_ambiente()
if MODELLO is None:
    MODELLO = scegli_modello(GroqLLM(chiave, "").elenca_modelli() or config.GROQ_MODELLI_PREFERITI)
llm = GroqLLM(chiave, MODELLO)
print("Groq | modello:", MODELLO)

## 7. Risposta e fonti
L'LLM riceve i passaggi etichettati `DOC_A`, `DOC_B`, … e deve restituire un JSON
`{"risposta": ..., "fonti": [...]}` vincolato da uno schema. Il codice scarta le etichette
inventate, converte le citazioni in [1], [2], … e associa a ognuna file e pagina.

In [ ]:
from rag.chat import rispondi_rag

r = rispondi_rag(DOMANDA_PROVA, indice, llm)
print(r.testo)
for f in r.fonti:
    print(f"[{f['id']}] {f['file']}, pagina {f['pagina']}")

## 8. Chat
Scrivere `esci` per terminare. Ogni domanda è indipendente dalle precedenti.

In [ ]:
print("CHATBOT RAG — scrivi 'esci' per terminare")
try:
    while True:
        domanda = input("\nTu: ").strip()
        if domanda.lower() == "esci":
            break
        if not domanda:
            continue
        r = rispondi_rag(domanda, indice, llm)
        print("\nChatbot:", r.testo)
        for f in r.fonti:
            print(f"[{f['id']}] {f['file']}, pagina {f['pagina']}")
except (KeyboardInterrupt, EOFError):
    pass
print("Chat terminata.")